### RAG Pipeline- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\PROJECT\AppData\Local\Temp\ipykernel_19680\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

                all_documents.extend(documents)
                print(f" ✓ Loaded {len(documents)} pages")

        except Exception as e:
                    print(f" ✗ Error : {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Java.pdf
 ✓ Loaded 1 pages

Processing: Python.pdf
 ✓ Loaded 1 pages

Total documents loaded: 2


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-10-24T13:03:07+00:00', 'author': 'PROJECT', 'moddate': '2026-10-24T13:03:07+00:00', 'source': '..\\data\\pdf\\Java.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Java.pdf', 'file_type': 'pdf'}, page_content='What is Java? \nJava is a popular and powerful programming language, created in 1995.  \nIt is owned by Oracle, and more than 3 billion devices run Java. \nIt is used for: \n\uf0b7 Mobile applications (specially Android apps)  \n\uf0b7 Desktop applications \n\uf0b7 Web applications \n\uf0b7 Web servers and application servers  \n\uf0b7 Games \n\uf0b7 Database connection \n\uf0b7 And much, much more! \nWhy Use Java? \n\uf0b7 Java works on different platforms (Windows, Mac, Linux, Raspberry Pi, etc.)  \n\uf0b7 It is one of the most popular programming languages in the world  \n\uf0b7 It has a large demand in the current job market \n\uf0b7 It is easy to 

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 2 documents into 5 chunks

Example chunk:
Content: What is Java? 
Java is a popular and powerful programming language, created in 1995.  
It is owned by Oracle, and more than 3 billion devices run Java. 
It is used for: 
 Mobile applications (special...
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-10-24T13:03:07+00:00', 'author': 'PROJECT', 'moddate': '2026-10-24T13:03:07+00:00', 'source': '..\\data\\pdf\\Java.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Java.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-10-24T13:03:07+00:00', 'author': 'PROJECT', 'moddate': '2026-10-24T13:03:07+00:00', 'source': '..\\data\\pdf\\Java.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Java.pdf', 'file_type': 'pdf'}, page_content='What is Java? \nJava is a popular and powerful programming language, created in 1995.  \nIt is owned by Oracle, and more than 3 billion devices run Java. \nIt is used for: \n\uf0b7 Mobile applications (specially Android apps)  \n\uf0b7 Desktop applications \n\uf0b7 Web applications \n\uf0b7 Web servers and application servers  \n\uf0b7 Games \n\uf0b7 Database connection \n\uf0b7 And much, much more! \nWhy Use Java? \n\uf0b7 Java works on different platforms (Windows, Mac, Linux, Raspberry Pi, etc.)  \n\uf0b7 It is one of the most popular programming languages in the world  \n\uf0b7 It has a large demand in the current job market \n\uf0b7 It is easy to 

### embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager             

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1906.15it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [21]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """

        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise        

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match exact number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise       


vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: <bound method Collection.count of Collection(name=pdf_documents)>


In [22]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-10-24T13:03:07+00:00', 'author': 'PROJECT', 'moddate': '2026-10-24T13:03:07+00:00', 'source': '..\\data\\pdf\\Java.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Java.pdf', 'file_type': 'pdf'}, page_content='What is Java? \nJava is a popular and powerful programming language, created in 1995.  \nIt is owned by Oracle, and more than 3 billion devices run Java. \nIt is used for: \n\uf0b7 Mobile applications (specially Android apps)  \n\uf0b7 Desktop applications \n\uf0b7 Web applications \n\uf0b7 Web servers and application servers  \n\uf0b7 Games \n\uf0b7 Database connection \n\uf0b7 And much, much more! \nWhy Use Java? \n\uf0b7 Java works on different platforms (Windows, Mac, Linux, Raspberry Pi, etc.)  \n\uf0b7 It is one of the most popular programming languages in the world  \n\uf0b7 It has a large demand in the current job market \n\uf0b7 It is easy to 

In [23]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector database
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 5 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]


Generated embeddings with shape: (5, 384)
Adding 5 documents to vector store...
Successfully added 5 documents to vector store
Total documents in collection: 5295


### Retriever Pipeline from VectorStore

In [24]:

from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        try:
            # 1. Generate query embedding
            query_embedding = self.embedding_manager.generate_embeddings(
                [query]
            )[0]

            # 2. Search in the vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # 3. Process results
            retrieved_docs = []

            documents = results.get("documents") or [[]]
            metadatas = results.get("metadatas") or [[]]
            distances = results.get("distances") or [[]]
            ids = results.get("ids") or [[]]

            if documents[0]:
                for rank, (doc_id, document, metadata, distance) in enumerate(
                    zip(
                        ids[0],
                        documents[0],
                        metadatas[0],
                        distances[0]
                    ),
                    start=1
                ):
                    similarity_score = 1.0 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": rank
                        })

            print(
                f"Retrieved {len(retrieved_docs)} documents "
                "(after filtering)"
            )
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Initialize the retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


In [25]:
rag_retriever

In [26]:
rag_retriever.retrieve("Why Python")

Retrieving documents for query: 'Why Python'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.63it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_eecdada6_4',
  'content': 'Python Syntax compared to other programming languages \n\uf0b7 Python was designed for readability, and has some similarities to the English language with influence from \nmathematics. \n\uf0b7 Python uses new lines to complete a command, as opposed to other programming langu ages which often use \nsemicolons or parentheses. \n\uf0b7 Python relies on indentation, using whitespace, to define scope; such as the scope of loops, functions and classes. \nOther programming languages often use curly -brackets for this purpose.',
  'metadata': {'page': 0,
   'total_pages': 1,
   'page_label': '1',
   'author': 'PROJECT',
   'doc_index': 4,
   'creationdate': '2026-10-24T13:03:24+00:00',
   'content_length': 504,
   'creator': 'Microsoft® Word 2016',
   'file_type': 'pdf',
   'source_file': 'Python.pdf',
   'source': '..\\data\\pdf\\Python.pdf',
   'producer': 'www.ilovepdf.com',
   'moddate': '2026-10-24T13:03:24+00:00'},
  'similarity_score': 0.32788652

In [27]:
rag_retriever.retrieve("Java")



Retrieving documents for query: 'Java'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 68.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_fda471d1_0',
  'content': 'What is Java? \nJava is a popular and powerful programming language, created in 1995.  \nIt is owned by Oracle, and more than 3 billion devices run Java. \nIt is used for: \n\uf0b7 Mobile applications (specially Android apps)  \n\uf0b7 Desktop applications \n\uf0b7 Web applications \n\uf0b7 Web servers and application servers  \n\uf0b7 Games \n\uf0b7 Database connection \n\uf0b7 And much, much more! \nWhy Use Java? \n\uf0b7 Java works on different platforms (Windows, Mac, Linux, Raspberry Pi, etc.)  \n\uf0b7 It is one of the most popular programming languages in the world  \n\uf0b7 It has a large demand in the current job market \n\uf0b7 It is easy to learn and simple to use  \n\uf0b7 It is open-source and free \n\uf0b7 It is secure, fast and powerful \n\uf0b7 It has huge community support (tens of millions of developers)  \n\uf0b7 Java is an object oriented language which gives a clear structure to p rograms and allows code to be reused, lowerin

### Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)


llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['context'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer thwe question concisely.
        Context:
        {context}

        Question: {query}

        Answer: """

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.context


In [30]:
answer=rag_simple("What is Python?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Python?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question.


### Enhanced RAG Pipeline Features

In [31]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Python?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Python?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.46it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 
